# People Dimension: AI and technical workforce intensity

`tech_team1_worker_share` (Tech Team 1 workers / total employees) and
`ai_worker_share` (AI workers / total employees). Numerators are ETO
PARAT's LinkedIn-based workforce snapshot (~2024); the denominator is
Compustat FY2024 `emp` converted to headcount. The shares measure how much
of a firm's workforce carries technical and AI-specific skills.

Missing is marked, not dropped: every universe firm keeps its row, and a
share is NaN when the firm is not matched to PARAT or to Compustat FY2024,
or its employee count is missing or zero. How to treat the NaNs is decided
at index-composition time.

Depends on the shared caches written by `structured_features_setup.ipynb`
for the same `UNIVERSE`. Writes
`data_clean/indicators/<UNIVERSE>/people.parquet`.

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.indicators.common.io import load_cached_step, write_indicator
from src.indicators.structured_features import (
    INDICATOR,
    PEOPLE_FEATURES,
    PEOPLE_REASON_COLS,
    people_indicator,
)

SHARED = INDICATOR
UNIVERSE = "sp500"

inputs = load_cached_step(SHARED, "inputs", UNIVERSE)
if inputs is None:
    raise RuntimeError(
        "Shared cache not found. Run notebooks/01_ingest_clean/"
        f"structured_features_setup.ipynb with UNIVERSE = {UNIVERSE!r} first."
    )
print(f"{len(inputs)} listed {UNIVERSE} firms in the input table")

## Compute the indicator (missing marked as NaN)

In [ ]:
indicator = people_indicator(inputs)
_n_complete = int(inputs["people_complete"].sum())
print(f"people ({UNIVERSE}): {len(indicator)} firm rows, {_n_complete} with both shares defined")

_missing = inputs[~inputs["people_complete"]]
_no_eto = _missing["eto_id"].isna()
_no_wrds = _missing["gvkey"].isna()
_no_emp = _missing["gvkey"].notna() & (_missing["employees_wrds"].isna() | _missing["employees_wrds"].eq(0))
print(f"missing {len(_missing)}: not in ETO PARAT {int(_no_eto.sum())}, "
      f"no Compustat match {int((_no_wrds & ~_no_eto).sum())}, "
      f"employees missing/zero {int((_no_emp & ~_no_eto).sum())} (groups overlap)")
indicator.head()

## Sanity checks

Uniqueness of the join key and feature ranges. Unlike the Technology
shares, the worker shares are ratios across two different sources (LinkedIn
snapshot vs. consolidated Compustat headcount), so values above 1 are
possible; they are reported here but left untouched — winsorizing or
capping is the job of `src/index/normalize.py`, not the indicator. The
`_reason` breakdown separates "unmatched" (no PARAT and/or no Compustat
record) from "zero_denominator" (matched, but zero/missing employees).

In [ ]:
assert indicator["normalized_company_name"].is_unique
for _col in PEOPLE_FEATURES:
    assert (indicator[_col].dropna() >= 0).all(), f"{_col} below 0"

_gt1 = indicator[(indicator[PEOPLE_FEATURES] > 1).any(axis=1)]
print(f"worker share > 1 (reported, not capped): {len(_gt1)}")
if len(_gt1):
    print(_gt1[["ticker", "company_name"] + PEOPLE_FEATURES].to_string(index=False))

_gt05 = indicator[(indicator[PEOPLE_FEATURES] > 0.5).any(axis=1)]
print(f"worker share > 0.5 (soft warning): {len(_gt05)}")
if len(_gt05):
    print(_gt05[["ticker", "company_name"] + PEOPLE_FEATURES].to_string(index=False))

print("\n=== Feature distributions ===")
print(indicator[PEOPLE_FEATURES].describe().to_string())

print("\n=== Missing-value reasons ===")
# Only the rows where the share is actually missing -- excluding those makes
# the breakdown read as "why is this one missing" instead of mixing in a
# confusing NaN count for the (majority) rows where nothing is wrong.
for _col in PEOPLE_REASON_COLS:
    _reasons = indicator[_col].dropna()
    print(f"\n{_col}: {len(_reasons)} missing of {len(indicator)}")
    print(_reasons.value_counts().to_string() if len(_reasons) else "  (none)")

## Write the indicator parquet

Only the id columns plus the two ratio features: `compose_index` treats
every non-id column as a feature, so raw counts and match provenance stay
in the `structured_features` cache and the HTML review.

In [ ]:
path = write_indicator(indicator, "people", UNIVERSE)
print(f"people ({UNIVERSE}): {len(indicator)} firm rows -> {path}")
indicator.head()